In [9]:
import torch
import torch.nn as nn 
import torch.nn.functional as F

In [14]:
! pip install timm 

  Using cached timm-1.0.19-py3-none-any.whl.metadata (60 kB)
Using cached timm-1.0.19-py3-none-any.whl (2.5 MB)


In [15]:
from timm.models.swin_transformer import window_partition,window_reverse

In [10]:
import torchtune

In [16]:
class swin_rotary_attention(nn.Module):
    def __init__(self,embed_dim,num_head):
        super().__init__()
        self.model = torchtune.modules.MultiHeadAttention(
        embed_dim = embed_dim, num_heads = num_head, num_kv_heads = num_head, head_dim = embed_dim//num_head,
        q_proj = nn.Linear(embed_dim,embed_dim),
        k_proj = nn.Linear(embed_dim,embed_dim),
        v_proj = nn.Linear(embed_dim,embed_dim),
        output_proj = nn.Linear(embed_dim,embed_dim),
        pos_embeddings = torchtune.modules.RotaryPositionalEmbeddings(dim= embed_dim//num_head,)
        )
    def forward(self,x):
        b,c,h,w,= x.shape
        x = x.permute(0,2,3,1)
        window_x = window_partition(x,[2,2]).reshape(-1,4,c)
        return  window_reverse(self.model(window_x,window_x),[2,2],h,w).permute(0,3,1,2)

        

In [21]:
swin_rotary_attention(256,2)(torch.rand(15,256,64,64)).shape

torch.Size([15, 256, 64, 64])

In [22]:
! pip install soft-moe


  Using cached soft_moe-0.0.1-py3-none-any.whl.metadata (4.1 kB)
Using cached soft_moe-0.0.1-py3-none-any.whl (14 kB)


In [23]:
! pip install einops

  Using cached einops-0.8.1-py3-none-any.whl.metadata (13 kB)
Using cached einops-0.8.1-py3-none-any.whl (64 kB)


In [24]:
from einops import rearrange, repeat


In [25]:
from soft_moe import SoftMoELayerWrapper

class rotary_attention(nn.Module):
    def __init__(self,embed_dim,num_head):
        super().__init__()
        self.model = torchtune.modules.MultiHeadAttention(
        embed_dim = embed_dim, num_heads = num_head, num_kv_heads = num_head, head_dim = embed_dim//num_head,
        q_proj = nn.Linear(embed_dim,embed_dim),
        k_proj = nn.Linear(embed_dim,embed_dim),
        v_proj = nn.Linear(embed_dim,embed_dim),
        output_proj = nn.Linear(embed_dim,embed_dim),
        pos_embeddings = torchtune.modules.RotaryPositionalEmbeddings(dim= embed_dim//num_head,)
        )
    def forward(self,x):
        b,len,dim= x.shape
        return  self.model(x,x)

        

class moe_func(nn.Module):
    def __init__(self,dim,slot_per_expert,num_expert,num_head):
        super().__init__()
        self.out = SoftMoELayerWrapper(
            dim=dim,
            slots_per_expert=slot_per_expert,
            num_experts=num_expert,
            layer=rotary_attention,
            # nn.Linear arguments
            # embed_dim=128,
            # num_heads=2,
            embed_dim = dim, 
            num_head = num_head, 
        )
    def forward(self,x):
        b,c,h,w = x.shape
        x = x.reshape(b,c,h*w).permute(0,2,1)
        return self.out(x).permute(0,1,2).reshape(b,c,h,w)

In [27]:
moe_func(128,2,4,2)(torch.rand(3,128,64,64)).shape

torch.Size([3, 128, 64, 64])

In [28]:
import torch
import math
import torch.nn as nn
import torch.nn.functional as F
from einops import rearrange, repeat
class MambaVisionMixer(nn.Module):

    def __init__(self, dim, d_state=16, kernel_size=3):
        super().__init__()
        self.embed_dim = torchtune.modules.RotaryPositionalEmbeddings(dim=dim)
        self.d_state = d_state
        self.dt_rank = math.ceil(dim / 16)
        self.in_proj = nn.Linear(dim, dim)
        self.x_proj = nn.Linear(dim//2, self.dt_rank + self.d_state *2)
        self.conv1d_x = nn.Conv1d(dim//2, dim//2, kernel_size=kernel_size, padding= 'same', groups=dim//2)
        self.conv1d_z = nn.Conv1d(dim//2, dim//2, kernel_size=kernel_size, padding= 'same', groups=dim//2)
        self.dt_proj = nn.Linear(self.dt_rank, dim//2)
        # dt = torch.exp(torch.rand(self.dim//2) * (math.log(dt_max) - math.log(dt_min)) + math.log(dt_min))
        A_log = torch.log(repeat(torch.arange(1, self.d_state + 1), 'n -> d n', d=dim//2))
        self.A_log = nn.Parameter(A_log)
        self.D = nn.Parameter(torch.ones(dim//2))
        self.out_proj = nn.Linear(dim, dim)
    def forward(self, hidden_states):
        b,c,h,w = hidden_states.shape
        # print(hidden_states.reshape(b,c,-1).shape)
        hidden_states = hidden_states.reshape(b,c,h*w).permute(0,2,1).reshape(b,int((h*w)/2),2,c)
        
        hidden_states = self.embed_dim(hidden_states)
        hidden_states = hidden_states.reshape(b,h*w,c)
        xz = rearrange(self.in_proj(hidden_states), 'b l d -> b d l')
        x, z = xz.chunk(2, dim=1)
        A = -torch.exp(self.A_log)
        x = F.silu(self.conv1d_x(x))
        z = F.silu(self.conv1d_z(z))
        seqlen = hidden_states.shape[1]
        x_dbl = self.x_proj(rearrange(x, 'b d l -> (b l) d'))
        dt, B, C = torch.split(x_dbl, [self.dt_rank, self.d_state, self.d_state], dim=-1)
        dt = rearrange(self.dt_proj(dt), '(b l) d -> b d l', l=seqlen)
        B = rearrange(B, '(b l) dstate -> b dstate l', l=seqlen)
        C = rearrange(C, '(b l) dstate -> b dstate l', l=seqlen)
        x_ssm = selective_scan_fn(x, dt, A, B, C, self.D)
        hidden_states = rearrange(torch.cat([x_ssm, z], dim=1), 'b d l -> b l d')
        return self.out_proj(hidden_states)

In [29]:
MambaVisionMixer(512)(torch.rand(1,512,64,64))

NameError: name 'selective_scan_fn' is not defined